In [0]:
%pip install cryptography requests
dbutils.library.restartPython()

In [0]:
kalshi_api_key_id = dbutils.secrets.get("kalshi", "KALSHI_API_KEY_ID")
kalshi_private_key = dbutils.secrets.get("kalshi", "KALSHI_PRIVATE_KEY")

In [0]:
import time
import base64
import requests

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding


# ============================================================
# Kalshi API config
# ============================================================

REST_HOST = "https://external-api.kalshi.com"
API_PREFIX = "/trade-api/v2"

SECRET_SCOPE = "kalshi"
API_KEY_ID_SECRET = "KALSHI_API_KEY_ID"

# Preferred if you store the private key as base64.
# Fallback supports raw PEM.
PRIVATE_KEY_B64_SECRET = "KALSHI_PRIVATE_KEY_B64"
PRIVATE_KEY_PEM_SECRET = "KALSHI_PRIVATE_KEY"


# ============================================================
# Simple secret helpers
# ============================================================

def get_secret(scope: str, key: str, required: bool = True) -> str | None:
    """
    Safely retrieve a Databricks secret.
    Returns None if required=False and the secret does not exist.
    """
    try:
        value = dbutils.secrets.get(scope=scope, key=key)
        if required and not value:
            raise ValueError(f"Secret {scope}/{key} is empty.")
        return value
    except Exception as e:
        if required:
            raise RuntimeError(f"Could not retrieve required secret: {scope}/{key}") from e
        return None


def normalize_pem(value: str) -> str:
    """
    Normalize a PEM key stored in Databricks secrets.
    Handles:
      - accidental wrapping quotes
      - escaped newlines: \\n
      - Windows newlines
      - extra blank lines / whitespace
    """
    value = value.strip()

    if (value.startswith('"') and value.endswith('"')) or (
        value.startswith("'") and value.endswith("'")
    ):
        value = value[1:-1].strip()

    value = (
        value.replace("\\r\\n", "\n")
             .replace("\\n", "\n")
             .replace("\r\n", "\n")
             .replace("\r", "\n")
    )

    lines = [line.strip() for line in value.split("\n") if line.strip()]
    return "\n".join(lines) + "\n"


def load_kalshi_private_key():
    """
    Loads Kalshi private key from Databricks secrets.

    Preferred:
      KALSHI_PRIVATE_KEY_B64 = base64-encoded PEM text

    Fallback:
      KALSHI_PRIVATE_KEY = raw PEM text
    """
    private_key_b64 = get_secret(
        SECRET_SCOPE,
        PRIVATE_KEY_B64_SECRET,
        required=False
    )

    if private_key_b64:
        print("Using base64 private key secret.")
        pem_text = base64.b64decode(private_key_b64).decode("utf-8")
    else:
        print("Using raw PEM private key secret.")
        pem_text = get_secret(
            SECRET_SCOPE,
            PRIVATE_KEY_PEM_SECRET,
            required=True
        )

    pem_text = normalize_pem(pem_text)

    lines = pem_text.splitlines()
    print("Private key first line:", repr(lines[0]) if lines else None)
    print("Private key last line:", repr(lines[-1]) if lines else None)
    print("Private key line count:", len(lines))

    if "BEGIN PUBLIC KEY" in lines[0]:
        raise ValueError(
            "You stored a public key, not a private key. "
            "Kalshi authentication requires the private key."
        )

    if "BEGIN ENCRYPTED PRIVATE KEY" in lines[0]:
        raise ValueError(
            "The private key is encrypted. This cell expects an unencrypted PEM key. "
            "Either store the password separately and pass it to load_pem_private_key, "
            "or export an unencrypted private key for this demo."
        )

    if "BEGIN OPENSSH PRIVATE KEY" in lines[0]:
        raise ValueError(
            "This is an OpenSSH private key. Store/export the key as PEM instead, "
            "for example BEGIN PRIVATE KEY or BEGIN RSA PRIVATE KEY."
        )

    private_key = serialization.load_pem_private_key(
        pem_text.encode("utf-8"),
        password=None,
    )

    return private_key


# ============================================================
# Load secrets
# ============================================================

API_KEY_ID = get_secret(
    SECRET_SCOPE,
    API_KEY_ID_SECRET,
    required=True
)

private_key = load_kalshi_private_key()

print("Loaded Kalshi API key id length:", len(API_KEY_ID))
print("Loaded Kalshi private key object:", type(private_key).__name__)


# ============================================================
# Signing helpers
# ============================================================

def sign_request(private_key, timestamp_ms: str, method: str, path: str) -> str:
    """
    Kalshi signs:
      timestamp + HTTP_METHOD + path_without_query

    Example:
      1703123456789GET/trade-api/v2/portfolio/balance
    """
    path_without_query = path.split("?")[0]
    message = f"{timestamp_ms}{method.upper()}{path_without_query}".encode("utf-8")

    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH,
        ),
        hashes.SHA256(),
    )

    return base64.b64encode(signature).decode("utf-8")


def kalshi_get(path: str, params: dict | None = None, auth: bool = False):
    """
    path should include /trade-api/v2, for example:
      /trade-api/v2/markets
      /trade-api/v2/portfolio/balance
    """
    if not path.startswith(API_PREFIX):
        raise ValueError(f"path must start with {API_PREFIX}. Got: {path}")

    method = "GET"
    url = f"{REST_HOST}{path}"

    headers = {}

    if auth:
        timestamp_ms = str(int(time.time() * 1000))
        signature = sign_request(
            private_key=private_key,
            timestamp_ms=timestamp_ms,
            method=method,
            path=path,
        )

        headers = {
            "KALSHI-ACCESS-KEY": API_KEY_ID,
            "KALSHI-ACCESS-TIMESTAMP": timestamp_ms,
            "KALSHI-ACCESS-SIGNATURE": signature,
        }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=(5, 30),
    )

    print("URL:", response.url)
    print("Status:", response.status_code)

    if response.status_code >= 400:
        print("Response text:", response.text[:1000])
        response.raise_for_status()

    return response.json()


# ============================================================
# Smoke tests
# ============================================================

print("Testing unauthenticated Kalshi market request...")

markets = kalshi_get(
    path=f"{API_PREFIX}/markets",
    params={"limit": 1},
    auth=False,
)

print("Unauthenticated market request succeeded.")
print(markets)

In [0]:
# ============================================================
# Pull 2025 FBS College Football Schedule From ESPN Public API
# ============================================================

import requests
import pandas as pd
from datetime import datetime, timezone


ESPN_CFB_SCOREBOARD_URL = (
    "https://site.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
)

# ESPN group=80 is commonly used for FBS / Division I-A.
ESPN_FBS_GROUP = 80


def fetch_espn_cfb_scoreboard_date_range(
    start_date: str,
    end_date: str,
    group: int = ESPN_FBS_GROUP,
    limit: int = 1000,
) -> dict:
    """
    Fetch ESPN college football scoreboard for a date range.

    start_date/end_date format:
      YYYY-MM-DD

    ESPN dates param format:
      YYYYMMDD-YYYYMMDD
    """
    start = start_date.replace("-", "")
    end = end_date.replace("-", "")

    params = {
        "groups": group,
        "dates": f"{start}-{end}",
        "limit": limit,
    }

    response = requests.get(
        ESPN_CFB_SCOREBOARD_URL,
        params=params,
        timeout=(5, 30),
    )

    print("URL:", response.url)
    print("Status:", response.status_code)

    response.raise_for_status()
    return response.json()


def parse_espn_cfb_games(scoreboard_json: dict) -> pd.DataFrame:
    rows = []

    for event in scoreboard_json.get("events", []):
        competition = (event.get("competitions") or [{}])[0]
        competitors = competition.get("competitors") or []

        home = next((c for c in competitors if c.get("homeAway") == "home"), {})
        away = next((c for c in competitors if c.get("homeAway") == "away"), {})

        home_team = home.get("team") or {}
        away_team = away.get("team") or {}

        rows.append({
            "espn_event_id": event.get("id"),
            "name": event.get("name"),
            "short_name": event.get("shortName"),
            "date_utc": event.get("date"),
            "season_year": (event.get("season") or {}).get("year"),
            "season_type": (event.get("season") or {}).get("type"),
            "week": (event.get("week") or {}).get("number"),
            "status_type": ((competition.get("status") or {}).get("type") or {}).get("name"),
            "completed": ((competition.get("status") or {}).get("type") or {}).get("completed"),
            "neutral_site": competition.get("neutralSite"),
            "conference_competition": competition.get("conferenceCompetition"),
            "home_team": home_team.get("displayName"),
            "home_abbrev_espn": home_team.get("abbreviation"),
            "home_score": home.get("score"),
            "away_team": away_team.get("displayName"),
            "away_abbrev_espn": away_team.get("abbreviation"),
            "away_score": away.get("score"),
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["date_utc"] = pd.to_datetime(df["date_utc"], utc=True, errors="coerce")
        df = df.sort_values("date_utc")

    return df


# 2025 regular season + bowls/playoff window.
# Adjust as needed.
espn_raw_2025 = fetch_espn_cfb_scoreboard_date_range(
    start_date="2025-08-01",
    end_date="2026-01-31",
)

espn_cfb_games_2025_df = parse_espn_cfb_games(espn_raw_2025)

print("Games returned:", len(espn_cfb_games_2025_df))
display(espn_cfb_games_2025_df)

In [0]:
# ============================================================
# Generate Home-Team Kalshi Ticker Candidates
# ============================================================

import pandas as pd
import re


KALSHI_CFB_GAME_SERIES = "KXNCAAFGAME"


ESPN_TO_KALSHI_ABBR_OVERRIDES = {
    # Publicly verified / common conventions
    "UNC": "UNC",
    "TCU": "TCU",

    # Common ESPN -> Kalshi cleanup
    "SC": "SCAR",
    "NCSU": "NCST",
    "WIS": "WIS",
    "BOIS": "BSU",
    "OU": "OKLA",
    "OKLA": "OKLA",
    "IU": "IND",
    "TA&M": "TXAM",
    "TAMU": "TXAM",
    "NU": "NW",

    # Power / common FBS teams
    "ALA": "ALA",
    "UGA": "UGA",
    "OSU": "OSU",
    "MICH": "MICH",
    "TEX": "TEX",
    "LSU": "LSU",
    "CLEM": "CLEM",
    "ORE": "ORE",
    "ND": "ND",
    "PSU": "PSU",
    "FSU": "FSU",
    "TENN": "TENN",
    "USC": "USC",
    "OKST": "OKST",
    "TTU": "TTU",
    "UCLA": "UCLA",
    "MIA": "MIA",
    "VT": "VT",
    "UVA": "UVA",
    "GT": "GT",
    "PITT": "PITT",
    "WVU": "WVU",
    "MINN": "MINN",
    "IOWA": "IOWA",
    "ISU": "ISU",
    "KSU": "KSU",
    "KU": "KU",
    "UK": "UK",
    "ARK": "ARK",
    "AUB": "AUB",
    "MIZ": "MIZZ",
    "MISS": "MISS",
    "MSST": "MSST",
    "LOU": "LOU",
    "SYR": "SYR",
    "DUKE": "DUKE",
    "STAN": "STAN",
    "CAL": "CAL",
    "COLO": "COLO",
    "UTAH": "UTAH",
    "ASU": "ASU",
    "ARIZ": "ARIZ",
    "BAY": "BAY",
    "UCF": "UCF",
    "HOU": "HOU",
    "CIN": "CIN",
    "BYU": "BYU",
    "SMU": "SMU",
    "NEB": "NEB",
    "MD": "MD",
    "PUR": "PUR",
    "NW": "NW",
    "ILL": "ILL",
    "IND": "IND",
    "RUTG": "RUTG",
    "BC": "BC",
    "WAKE": "WAKE",
    "MEM": "MEM",
    "TULN": "TULN",
    "NAVY": "NAVY",
    "ARMY": "ARMY",
    "AF": "AF",
    "APP": "APP",
    "JMU": "JMU",
    "LIB": "LIB",
    "ODU": "ODU",
    "ECU": "ECU",
    "USF": "USF",
    "FAU": "FAU",
    "FIU": "FIU",
    "UNT": "UNT",
    "UTSA": "UTSA",
    "RICE": "RICE",
    "TULSA": "TULSA",
    "TEM": "TEM",
    "CHAR": "CHAR",
    "CLT": "CHAR",
    "CONN": "CONN",
    "UMASS": "UMASS",
    "HAW": "HAW",
    "SJSU": "SJSU",
    "SDSU": "SDSU",
    "FRES": "FRES",
    "NEV": "NEV",
    "UNLV": "UNLV",
    "USU": "USU",
    "WYO": "WYO",
    "CSU": "CSU",
    "UNM": "UNM",
    "UTEP": "UTEP",
    "NMSU": "NMSU",
    "MTSU": "MTU",
    "WKU": "WKU",
    "LT": "LT",
    "SHSU": "SHSU",
    "JVST": "JVST",
    "KENN": "KENN",
    "MRSH": "MRSH",
    "TROY": "TROY",
    "USA": "USA",
    "UL": "ULL",
    "ULM": "ULM",
    "ARKST": "ARST",
    "ARST": "ARST",
    "TXST": "TXST",
    "GASO": "GASO",
    "GAST": "GAST",
    "CCU": "CCAR",
    "BUFF": "BUFF",
    "AKR": "AKR",
    "KENT": "KENT",
    "BGSU": "BGSU",
    "TOL": "TOL",
    "NIU": "NIU",
    "EMU": "EMU",
    "CMU": "CMU",
    "WMU": "WMU",
    "BALL": "BALL",
    "OHIO": "OHIO",
    "M-OH": "M-OH",
    "MOST": "MOST",
    "NDSU": "NDSU",
    "SAC": "SAC",
}


def date_utc_to_kalshi_date_part(date_utc) -> str:
    ts = pd.to_datetime(date_utc, utc=True)
    eastern_ts = ts.tz_convert("America/New_York")
    return eastern_ts.strftime("%y%b%d").upper()


def clean_abbr_for_kalshi(abbr: str) -> str | None:
    if pd.isna(abbr):
        return None

    abbr = str(abbr).strip().upper()

    if abbr in ESPN_TO_KALSHI_ABBR_OVERRIDES:
        return ESPN_TO_KALSHI_ABBR_OVERRIDES[abbr]

    return re.sub(r"[^A-Z0-9]", "", abbr)


def generate_home_team_kalshi_market_ticker_candidates_from_espn_row(row) -> list[str]:
    date_part = date_utc_to_kalshi_date_part(row["date_utc"])

    away = clean_abbr_for_kalshi(row["away_abbrev_espn"])
    home = clean_abbr_for_kalshi(row["home_abbrev_espn"])

    if not away or not home:
        return []

    return [
        f"{KALSHI_CFB_GAME_SERIES}-{date_part}{away}{home}-{home}",
        f"{KALSHI_CFB_GAME_SERIES}-{date_part}{home}{away}-{home}",
    ]


cfb_games_with_candidates_df = espn_cfb_games_2025_df.copy()

cfb_games_with_candidates_df["kalshi_date_part"] = cfb_games_with_candidates_df["date_utc"].apply(
    date_utc_to_kalshi_date_part
)

cfb_games_with_candidates_df["away_abbrev_kalshi"] = cfb_games_with_candidates_df["away_abbrev_espn"].apply(
    clean_abbr_for_kalshi
)

cfb_games_with_candidates_df["home_abbrev_kalshi"] = cfb_games_with_candidates_df["home_abbrev_espn"].apply(
    clean_abbr_for_kalshi
)

cfb_games_with_candidates_df["kalshi_home_market_ticker_candidates"] = cfb_games_with_candidates_df.apply(
    generate_home_team_kalshi_market_ticker_candidates_from_espn_row,
    axis=1,
)

cfb_games_with_candidates_df["kalshi_home_market_candidate_1"] = cfb_games_with_candidates_df[
    "kalshi_home_market_ticker_candidates"
].apply(lambda x: x[0] if len(x) > 0 else None)

cfb_games_with_candidates_df["kalshi_home_market_candidate_2"] = cfb_games_with_candidates_df[
    "kalshi_home_market_ticker_candidates"
].apply(lambda x: x[1] if len(x) > 1 else None)

display(
    cfb_games_with_candidates_df[
        [
            "espn_event_id",
            "date_utc",
            "week",
            "away_team",
            "away_abbrev_espn",
            "away_abbrev_kalshi",
            "home_team",
            "home_abbrev_espn",
            "home_abbrev_kalshi",
            "kalshi_home_market_candidate_1",
            "kalshi_home_market_candidate_2",
        ]
    ]
)

In [0]:
# Known public current/future example from Kalshi URL:
# North Carolina vs TCU, Aug 29, 2026
# candidate_tickers = generate_kalshi_cfb_game_ticker_candidates(
#     game_date="2026-08-29",
#     team_a="North Carolina",
#     team_b="TCU",
# )

# candidate_tickers

In [0]:
# ============================================================
# CELL 2: Validate Home-Team Kalshi Market Tickers
# ============================================================

import time
import requests
import pandas as pd


TEST_LIMIT = 25
PERIOD_INTERVAL = 1440
SLEEP_SECONDS = 0.05
DAYS_BEFORE_GAME = 30
DAYS_AFTER_GAME = 3


def get_candle_window_around_game(
    date_utc,
    days_before: int = DAYS_BEFORE_GAME,
    days_after: int = DAYS_AFTER_GAME,
) -> tuple[int, int]:
    game_ts = pd.to_datetime(date_utc, utc=True)

    start_ts = int((game_ts - pd.Timedelta(days=days_before)).timestamp())
    end_ts = int((game_ts + pd.Timedelta(days=days_after)).timestamp())

    return start_ts, end_ts


def extract_candles_from_response(data: dict) -> list:
    return (
        data.get("candlesticks")
        or data.get("market_candlesticks")
        or data.get("candles")
        or []
    )


def test_historical_candlestick_ticker_quiet(
    ticker: str,
    start_ts: int,
    end_ts: int,
    period_interval: int = PERIOD_INTERVAL,
) -> dict:
    url = f"{REST_HOST}{API_PREFIX}/historical/markets/{ticker}/candlesticks"

    try:
        response = requests.get(
            url,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": period_interval,
            },
            timeout=(5, 30),
        )

        result = {
            "ticker": ticker,
            "endpoint_type": "historical",
            "url": response.url,
            "status_code": response.status_code,
            "exists": response.status_code == 200,
            "candles_count": 0,
            "response_preview": response.text[:500],
        }

        if response.status_code == 200:
            data = response.json()
            candles = extract_candles_from_response(data)
            result["candles_count"] = len(candles)

        return result

    except Exception as e:
        return {
            "ticker": ticker,
            "endpoint_type": "historical",
            "url": url,
            "status_code": None,
            "exists": False,
            "candles_count": 0,
            "response_preview": repr(e),
        }


def test_live_candlestick_ticker_quiet(
    ticker: str,
    start_ts: int,
    end_ts: int,
    period_interval: int = PERIOD_INTERVAL,
    series_ticker: str = KALSHI_CFB_GAME_SERIES,
) -> dict:
    url = f"{REST_HOST}{API_PREFIX}/series/{series_ticker}/markets/{ticker}/candlesticks"

    try:
        response = requests.get(
            url,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": period_interval,
            },
            timeout=(5, 30),
        )

        result = {
            "ticker": ticker,
            "endpoint_type": "live_recent",
            "url": response.url,
            "status_code": response.status_code,
            "exists": response.status_code == 200,
            "candles_count": 0,
            "response_preview": response.text[:500],
        }

        if response.status_code == 200:
            data = response.json()
            candles = extract_candles_from_response(data)
            result["candles_count"] = len(candles)

        return result

    except Exception as e:
        return {
            "ticker": ticker,
            "endpoint_type": "live_recent",
            "url": url,
            "status_code": None,
            "exists": False,
            "candles_count": 0,
            "response_preview": repr(e),
        }


def find_valid_home_team_kalshi_ticker_for_game(
    row,
    sleep_seconds: float = SLEEP_SECONDS,
    try_live_fallback: bool = True,
) -> dict:
    candidates = row.get("kalshi_home_market_ticker_candidates") or []
    start_ts, end_ts = get_candle_window_around_game(row["date_utc"])

    attempts = []

    for candidate in candidates:
        historical_result = test_historical_candlestick_ticker_quiet(
            ticker=candidate,
            start_ts=start_ts,
            end_ts=end_ts,
            period_interval=PERIOD_INTERVAL,
        )

        attempts.append(historical_result)

        if sleep_seconds:
            time.sleep(sleep_seconds)

        if historical_result["exists"]:
            return {
                "espn_event_id": row["espn_event_id"],
                "date_utc": row["date_utc"],
                "away_team": row["away_team"],
                "away_abbrev_kalshi": row["away_abbrev_kalshi"],
                "home_team": row["home_team"],
                "home_abbrev_kalshi": row["home_abbrev_kalshi"],
                "kalshi_ticker_found": True,
                "kalshi_market_perspective": "home_team_yes",
                "kalshi_ticker": candidate,
                "kalshi_endpoint_type": historical_result["endpoint_type"],
                "kalshi_validation_status_code": historical_result["status_code"],
                "kalshi_candles_count": historical_result["candles_count"],
                "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
                "kalshi_attempt_results": attempts,
            }

        if try_live_fallback:
            live_result = test_live_candlestick_ticker_quiet(
                ticker=candidate,
                start_ts=start_ts,
                end_ts=end_ts,
                period_interval=PERIOD_INTERVAL,
                series_ticker=KALSHI_CFB_GAME_SERIES,
            )

            attempts.append(live_result)

            if sleep_seconds:
                time.sleep(sleep_seconds)

            if live_result["exists"]:
                return {
                    "espn_event_id": row["espn_event_id"],
                    "date_utc": row["date_utc"],
                    "away_team": row["away_team"],
                    "away_abbrev_kalshi": row["away_abbrev_kalshi"],
                    "home_team": row["home_team"],
                    "home_abbrev_kalshi": row["home_abbrev_kalshi"],
                    "kalshi_ticker_found": True,
                    "kalshi_market_perspective": "home_team_yes",
                    "kalshi_ticker": candidate,
                    "kalshi_endpoint_type": live_result["endpoint_type"],
                    "kalshi_validation_status_code": live_result["status_code"],
                    "kalshi_candles_count": live_result["candles_count"],
                    "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
                    "kalshi_attempt_results": attempts,
                }

    return {
        "espn_event_id": row["espn_event_id"],
        "date_utc": row["date_utc"],
        "away_team": row["away_team"],
        "away_abbrev_kalshi": row["away_abbrev_kalshi"],
        "home_team": row["home_team"],
        "home_abbrev_kalshi": row["home_abbrev_kalshi"],
        "kalshi_ticker_found": False,
        "kalshi_market_perspective": "home_team_yes",
        "kalshi_ticker": None,
        "kalshi_endpoint_type": None,
        "kalshi_validation_status_code": None,
        "kalshi_candles_count": 0,
        "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
        "kalshi_attempt_results": attempts,
    }


source_df = cfb_games_with_candidates_df.copy()

if TEST_LIMIT is not None:
    source_df = source_df.head(TEST_LIMIT)

validation_rows = []

for idx, row in source_df.iterrows():
    game_num = len(validation_rows) + 1

    print(
        f"[{game_num}/{len(source_df)}] "
        f"HOME YES market: {row['away_team']} at {row['home_team']} "
        f"({row['date_utc']})"
    )

    print("  candidates:", row.get("kalshi_home_market_ticker_candidates"))

    validation = find_valid_home_team_kalshi_ticker_for_game(
        row=row,
        sleep_seconds=SLEEP_SECONDS,
        try_live_fallback=True,
    )

    validation_rows.append(validation)

    if validation["kalshi_ticker_found"]:
        print(
            f"  FOUND: {validation['kalshi_ticker']} "
            f"via {validation['kalshi_endpoint_type']} "
            f"candles={validation['kalshi_candles_count']}"
        )
    else:
        print("  not found")
        print("  attempted:", validation["kalshi_attempted_tickers"])

kalshi_home_ticker_validation_df = pd.DataFrame(validation_rows)

print("\nValidation complete.")
print("Games checked:", len(kalshi_home_ticker_validation_df))
print("Tickers found:", kalshi_home_ticker_validation_df["kalshi_ticker_found"].fillna(False).sum())

display(kalshi_home_ticker_validation_df)

In [0]:
# ============================================================
# Validate Home-Team Kalshi Market Tickers for 2025 CFB Games
# ============================================================

import time
import requests
import pandas as pd


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

# Start small while debugging. Change to None for the full season.
TEST_LIMIT = None

# Use daily candles for ticker validation.
# Valid Kalshi intervals: 1, 60, 1440
PERIOD_INTERVAL = 1440

# Small delay to avoid hammering the API.
SLEEP_SECONDS = 0.05

# Validation window around each game.
# We only need a small window to prove the ticker exists.
DAYS_BEFORE_GAME = 30
DAYS_AFTER_GAME = 3


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def get_candle_window_around_game(
    date_utc,
    days_before: int = DAYS_BEFORE_GAME,
    days_after: int = DAYS_AFTER_GAME,
) -> tuple[int, int]:
    """
    Build a timestamp window around the ESPN game date.

    This is only used to validate whether the Kalshi ticker exists
    and has historical candlesticks.
    """
    game_ts = pd.to_datetime(date_utc, utc=True)

    start_ts = int((game_ts - pd.Timedelta(days=days_before)).timestamp())
    end_ts = int((game_ts + pd.Timedelta(days=days_after)).timestamp())

    return start_ts, end_ts


def extract_candles_from_response(data: dict) -> list:
    """
    Kalshi response shapes can vary slightly depending on endpoint/version.
    This safely checks common candle container names.
    """
    return (
        data.get("candlesticks")
        or data.get("market_candlesticks")
        or data.get("candles")
        or []
    )


def test_historical_candlestick_ticker_quiet(
    ticker: str,
    start_ts: int,
    end_ts: int,
    period_interval: int = PERIOD_INTERVAL,
) -> dict:
    """
    Test whether a Kalshi historical candlestick ticker exists.

    Does not raise on 404.
    Returns diagnostics so we can inspect failures later.
    """
    url = f"{REST_HOST}{API_PREFIX}/historical/markets/{ticker}/candlesticks"

    try:
        response = requests.get(
            url,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": period_interval,
            },
            timeout=(5, 30),
        )

        result = {
            "ticker": ticker,
            "endpoint_type": "historical",
            "url": response.url,
            "status_code": response.status_code,
            "exists": response.status_code == 200,
            "candles_count": 0,
            "response_preview": response.text[:500],
        }

        if response.status_code == 200:
            data = response.json()
            candles = extract_candles_from_response(data)
            result["candles_count"] = len(candles)

        return result

    except Exception as e:
        return {
            "ticker": ticker,
            "endpoint_type": "historical",
            "url": url,
            "status_code": None,
            "exists": False,
            "candles_count": 0,
            "response_preview": repr(e),
        }


def test_live_candlestick_ticker_quiet(
    ticker: str,
    start_ts: int,
    end_ts: int,
    period_interval: int = PERIOD_INTERVAL,
    series_ticker: str = "KXNCAAFGAME",
) -> dict:
    """
    Fallback test against the live/recent candlestick endpoint.

    Useful because some 2025/2026 markets may not yet be in the historical tier,
    depending on Kalshi's cutoff.
    """
    url = f"{REST_HOST}{API_PREFIX}/series/{series_ticker}/markets/{ticker}/candlesticks"

    try:
        response = requests.get(
            url,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": period_interval,
            },
            timeout=(5, 30),
        )

        result = {
            "ticker": ticker,
            "endpoint_type": "live_recent",
            "url": response.url,
            "status_code": response.status_code,
            "exists": response.status_code == 200,
            "candles_count": 0,
            "response_preview": response.text[:500],
        }

        if response.status_code == 200:
            data = response.json()
            candles = extract_candles_from_response(data)
            result["candles_count"] = len(candles)

        return result

    except Exception as e:
        return {
            "ticker": ticker,
            "endpoint_type": "live_recent",
            "url": url,
            "status_code": None,
            "exists": False,
            "candles_count": 0,
            "response_preview": repr(e),
        }


def find_valid_home_team_kalshi_ticker_for_game(
    row,
    sleep_seconds: float = SLEEP_SECONDS,
    try_live_fallback: bool = True,
) -> dict:
    """
    Try home-team YES market ticker candidates for one ESPN game.

    Standard convention:
      One ESPN game -> one Kalshi market ticker
      Market perspective -> home team YES

    Candidate examples:
      KXNCAAFGAME-25AUG30AWAYHOME-HOME
      KXNCAAFGAME-25AUG30HOMEAWAY-HOME
    """
    candidates = row.get("kalshi_home_market_ticker_candidates") or []
    start_ts, end_ts = get_candle_window_around_game(row["date_utc"])

    attempts = []

    for candidate in candidates:
        # First try the historical endpoint.
        historical_result = test_historical_candlestick_ticker_quiet(
            ticker=candidate,
            start_ts=start_ts,
            end_ts=end_ts,
            period_interval=PERIOD_INTERVAL,
        )

        attempts.append(historical_result)

        if sleep_seconds:
            time.sleep(sleep_seconds)

        if historical_result["exists"]:
            return {
                "espn_event_id": row["espn_event_id"],
                "date_utc": row["date_utc"],
                "away_team": row["away_team"],
                "away_abbrev_kalshi": row["away_abbrev_kalshi"],
                "home_team": row["home_team"],
                "home_abbrev_kalshi": row["home_abbrev_kalshi"],
                "kalshi_ticker_found": True,
                "kalshi_market_perspective": "home_team_yes",
                "kalshi_ticker": candidate,
                "kalshi_endpoint_type": historical_result["endpoint_type"],
                "kalshi_validation_status_code": historical_result["status_code"],
                "kalshi_candles_count": historical_result["candles_count"],
                "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
                "kalshi_attempt_results": attempts,
            }

        # If historical fails, optionally try live/recent endpoint too.
        if try_live_fallback:
            live_result = test_live_candlestick_ticker_quiet(
                ticker=candidate,
                start_ts=start_ts,
                end_ts=end_ts,
                period_interval=PERIOD_INTERVAL,
                series_ticker="KXNCAAFGAME",
            )

            attempts.append(live_result)

            if sleep_seconds:
                time.sleep(sleep_seconds)

            if live_result["exists"]:
                return {
                    "espn_event_id": row["espn_event_id"],
                    "date_utc": row["date_utc"],
                    "away_team": row["away_team"],
                    "away_abbrev_kalshi": row["away_abbrev_kalshi"],
                    "home_team": row["home_team"],
                    "home_abbrev_kalshi": row["home_abbrev_kalshi"],
                    "kalshi_ticker_found": True,
                    "kalshi_market_perspective": "home_team_yes",
                    "kalshi_ticker": candidate,
                    "kalshi_endpoint_type": live_result["endpoint_type"],
                    "kalshi_validation_status_code": live_result["status_code"],
                    "kalshi_candles_count": live_result["candles_count"],
                    "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
                    "kalshi_attempt_results": attempts,
                }

    return {
        "espn_event_id": row["espn_event_id"],
        "date_utc": row["date_utc"],
        "away_team": row["away_team"],
        "away_abbrev_kalshi": row["away_abbrev_kalshi"],
        "home_team": row["home_team"],
        "home_abbrev_kalshi": row["home_abbrev_kalshi"],
        "kalshi_ticker_found": False,
        "kalshi_market_perspective": "home_team_yes",
        "kalshi_ticker": None,
        "kalshi_endpoint_type": None,
        "kalshi_validation_status_code": None,
        "kalshi_candles_count": 0,
        "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
        "kalshi_attempt_results": attempts,
    }


# ------------------------------------------------------------
# Run validation loop
# ------------------------------------------------------------

source_df = cfb_games_with_candidates_df.copy()

if TEST_LIMIT is not None:
    source_df = source_df.head(TEST_LIMIT)

validation_rows = []

for idx, row in source_df.iterrows():
    game_num = len(validation_rows) + 1

    print(
        f"[{game_num}/{len(source_df)}] "
        f"HOME YES market: {row['away_team']} at {row['home_team']} "
        f"({row['date_utc']})"
    )

    print("  candidates:", row.get("kalshi_home_market_ticker_candidates"))

    validation = find_valid_home_team_kalshi_ticker_for_game(
        row=row,
        sleep_seconds=SLEEP_SECONDS,
        try_live_fallback=True,
    )

    validation_rows.append(validation)

    if validation["kalshi_ticker_found"]:
        print(
            f"  FOUND: {validation['kalshi_ticker']} "
            f"via {validation['kalshi_endpoint_type']} "
            f"candles={validation['kalshi_candles_count']}"
        )
    else:
        print("  not found")
        print("  attempted:", validation["kalshi_attempted_tickers"])

kalshi_home_ticker_validation_df = pd.DataFrame(validation_rows)

print("\nValidation complete.")
print("Games checked:", len(kalshi_home_ticker_validation_df))
print("Tickers found:", kalshi_home_ticker_validation_df["kalshi_ticker_found"].fillna(False).sum())

display(kalshi_home_ticker_validation_df)

In [0]:
# ============================================================
# Join Validated Home-Team Kalshi Tickers Back to Games
# ============================================================

cfb_games_home_kalshi_ticker_df = cfb_games_with_candidates_df.merge(
    kalshi_home_ticker_validation_df[
        [
            "espn_event_id",
            "kalshi_ticker_found",
            "kalshi_market_perspective",
            "kalshi_ticker",
            "kalshi_endpoint_type",
            "kalshi_candles_count",
            "kalshi_attempted_tickers",
            "kalshi_attempt_results",
        ]
    ],
    on="espn_event_id",
    how="left",
)

cfb_games_home_kalshi_ticker_df["kalshi_ticker_found"] = (
    cfb_games_home_kalshi_ticker_df["kalshi_ticker_found"]
    .fillna(False)
    .astype(bool)
)

print("Total games:", len(cfb_games_home_kalshi_ticker_df))
print("Home-team tickers found:", cfb_games_home_kalshi_ticker_df["kalshi_ticker_found"].sum())

display(
    cfb_games_home_kalshi_ticker_df[
        [
            "espn_event_id",
            "date_utc",
            "week",
            "away_team",
            "away_abbrev_espn",
            "away_abbrev_kalshi",
            "home_team",
            "home_abbrev_espn",
            "home_abbrev_kalshi",
            "kalshi_home_market_candidate_1",
            "kalshi_home_market_candidate_2",
            "kalshi_ticker_found",
            "kalshi_market_perspective",
            "kalshi_ticker",
            "kalshi_endpoint_type",
            "kalshi_candles_count",
        ]
    ]
)